# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

## STEP 1

### Make relevant library imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re 

## STEP 2

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

#### 1. Loading and previewing the dataset

In [84]:
av_data = pd.read_csv("data/AviationData.csv" , encoding="latin1")

# preview first 5 rows
av_data.head(5)

C:\Users\hawan\AppData\Local\Temp\ipykernel_15976\3462321126.py:1: DtypeWarning: Columns (0: Latitude, 1: Longitude, 2: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  av_data = pd.read_csv("data/AviationData.csv" , encoding="latin1")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


#### 2. Check Datatypes, missing values and summary statistics 

In [ ]:
# Col names, datatypes and non_null counts
av_data.info()

# Number of missing values
av_data.isnull().sum().sort_values(ascending=False)

# Rows and cols 
av_data.shape 

## STEP 3

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

### 1. Checking the aircraft category

In [ ]:
# Aircraft category value counts
print(av_data['Aircraft.Category'].value_counts(dropna=False))
print(av_data['Amateur.Built'].value_counts(dropna=False))



Note: There are 56,602 null values for Aircraft.Category and 102 missing values for Amateur.Built

In [ ]:
# Which years does this missingness concentrate around 
av_data['year'] = pd.to_datetime(av_data['Event.Date']).dt.year
av_data.groupby('year')['Aircraft.Category'].apply(lambda x: x.isna().mean())*100

Note: Upon further inspection we find that most missing values for Aircraft.Category are from 1983 to 2007 whereas after 2008 the missingness frops to almost 0%

### 2. Cleaning by filtering our data to keep Professional Aircraft since 1983 to 2023 

1. Before 2007 the field  Aircraft category was not being filled.
There is a 99% missing value rate between 1983 and 2007 while from 2008 onwards this drops to almost 0%
Dataset would hence be skewed towards 2008-2023.

In [ ]:
#  1. Aircraft.Category: keep only Airplanes 
av_data = av_data[av_data['Aircraft.Category'] == 'Airplane']

#  2. Amateur.Built: keep only professional builds 
# Only 102 NaNs out of ~88k rows -- negligible, 
av_data = av_data[av_data['Amateur.Built'] == 'No']

# --- 3. Event.Date: keep only accidents from 1983 onward to account for the 40 years requirement
av_data['Event.Date'] = pd.to_datetime(av_data['Event.Date'])
av_data = av_data[av_data['Event.Date'] >= '1983-01-01']

print(av_data.shape)

## STEP 4

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [ ]:
# Inspect
inj_cols = ['Total.Fatal.Injuries','Total.Serious.Injuries','Total.Minor.Injuries','Total.Uninjured']
print(av_data[inj_cols].isna().sum())
print('Rows with all 4 missing:', av_data[inj_cols].isna().all(axis=1).sum())

filled = av_data[inj_cols].fillna(0)
print('Rows summing to 0 passengers:', (filled.sum(axis=1) == 0).sum())

Note: There are rows with completely missing values for all 4 columns and columns with a total of 0 passengers 

In [ ]:
inj_cols = ['Total.Fatal.Injuries','Total.Serious.Injuries','Total.Minor.Injuries','Total.Uninjured']

# Drop rows with zero info on any injuries -- can't estimate passengers at all
av_data = av_data.dropna(subset=inj_cols, how='all')

# Assumption: remaining NaNs in these columns mean "0 of this injury type", not "unknown"
av_data[inj_cols] = av_data[inj_cols].fillna(0)

# Estimate total passengers aboard
av_data['Total.Passengers'] = av_data[inj_cols].sum(axis=1)

# Drop flights where no passengers are recorded at all 
av_data = av_data[av_data['Total.Passengers'] > 0]

# Derived metric: fraction of passengers fatally or seriously injured
av_data['Injury.Fraction'] = (
    (av_data['Total.Fatal.Injuries'] + av_data['Total.Serious.Injuries'])
    / av_data['Total.Passengers']
)

print(av_data['Injury.Fraction'].describe())

2. Handled Injury columns by dropping 47 rows that had missing values all the columns and filled in 0 for the remaining columns with null values. Dropped 904 rows that also recorded 0 passengers 

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

### Drop the null and 'Unknown' values for aircraft damage 

In [ ]:
# Inspect
print(av_data['Aircraft.damage'].value_counts(dropna=False))

The Null values are 1227 and Unknown values 97.

In [ ]:
# Drop rows where damage outcome isn't actually known
av_data = av_data[~av_data['Aircraft.damage'].isin(['Unknown']) & av_data['Aircraft.damage'].notna()]

# Derived binary column: was the aircraft destroyed?
av_data['Is.Destroyed'] = (av_data['Aircraft.damage'] == 'Destroyed').astype(int)

print(av_data['Is.Destroyed'].value_counts(normalize=True))

# Drop rows where damage outcome isn't actually known
av_data = av_data[~av_data['Aircraft.damage'].isin(['Unknown']) & av_data['Aircraft.damage'].notna()]

# Derived binary column: was the aircraft destroyed?
av_data['Is.Destroyed'] = (av_data['Aircraft.damage'] == 'Destroyed').astype(int)

print(av_data['Is.Destroyed'].value_counts(normalize=True))

av_data.head(15)

3. Dropped missing and unknown values for Aircraft damage which constituted to about 6% of the total rows.
As for the Is.Destroyed column the airplanes that are Destroyed are marked as binary 1 and the Substantial and Minor as binary 0

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [ ]:
# Inspect
print('Unique raw Makes:', av_data['Make'].nunique())
print('Unique after upper+strip:', av_data['Make'].str.upper().str.strip().nunique())
print(av_data['Make'].str.upper().str.strip().value_counts().head(30))

4.  Cleaning tasks for 'Make'
Standardize casing and strip whitespace (.upper().strip()), Remove punctuation (periods/commas) that create near-duplicates (CORP. vs CORP), 
Strip common corporate suffixes (INC, CORP, CORPORATION, CO, COMPANY, LLC), so entity-name variants of the same manufacturer collapse together, 
Fix a few known spelling/spacing aliases (DEHAVILLAND → DE HAVILLAND), 
Drop the 1 row with Make NaN (negligible), 
Apply the ≥50-record threshold for the final analysis set

In [ ]:
def clean_make(m):
    if pd.isna(m):
        return m
    m = m.upper().strip()
    m = re.sub(r'[.,]', '', m)  # drop periods/commas: "CORP." -> "CORP"
    
    # strip trailing corporate-entity suffixes so name variants merge
    suffixes = [' CORPORATION', ' COMPANY', ' CORP', ' INC', ' CO', ' LLC', ' LTD']
    for suf in suffixes:
        if m.endswith(suf):
            m = m[: -len(suf)].strip()
            break
    
    # known spelling/spacing aliases
    aliases = {'DEHAVILLAND': 'DE HAVILLAND'}
    return aliases.get(m, m)

av_data = av_data.dropna(subset=['Make'])
av_data['Make'] = av_data['Make'].apply(clean_make)

make_counts = av_data['Make'].value_counts()
print(f"Unique makes before threshold: {len(make_counts)}")

# keep only makes with >=50 records for statistically reasonable comparisons
valid_makes = make_counts[make_counts >= 50].index
av_data = av_data[av_data['Make'].isin(valid_makes)]

print(f"Makes retained: {len(valid_makes)}")
print(av_data['Make'].value_counts())

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
# Inspect
print('Model NaNs:', av_data['Model'].isna().sum())
print('Unique Models:', av_data['Model'].nunique())

# check whether model labels are unique to a single make
dup_check = av_data.dropna(subset=['Model']).groupby('Model')['Make'].nunique().sort_values(ascending=False)
print(dup_check.head(10))

In [ ]:
# Drop the few rows with missing Model
av_data = av_data.dropna(subset=['Model'])

# Standardize formatting, same as Make
av_data['Model'] = av_data['Model'].str.upper().str.strip()

# Model labels collide across manufacturers e.g. "500" exists under 3+ makes,
# so build a combined identifier to uniquely represent a specific plane type
av_data['Make.Model'] = av_data['Make'] + ' ' + av_data['Model']

print(f"Unique Make.Model combos: {av_data['Make.Model'].nunique()}")
print(av_data['Make.Model'].value_counts().head(10))

5. Model cleaning. Drop all the Null values, labels are not unique and have to be paired with Make to define a plane type

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
# Inspect
for col in ['Engine.Type','Weather.Condition','Number.of.Engines','Purpose.of.flight','Broad.phase.of.flight']:
    print(f'=== {col} ===')
    print(av_data[col].value_counts(dropna=False))
    print()

6. For the rest of the  relevant columns 
EngineType and WeatherCondition UNK replaced with unknown to consolidate the two. 
Rows with null or 0.0 values for number of engines immediately dropped.
Purpose of flight formatted to consolidate Air Race Show and Air Race/show

In [ ]:
# Engine.Type: consolidate unknown labels 
av_data['Engine.Type'] = av_data['Engine.Type'].replace({'UNK': 'Unknown'})

# Weather.Condition: consolidate unknown labels
av_data['Weather.Condition'] = av_data['Weather.Condition'].replace({'Unk': 'Unknown', 'UNK': 'Unknown'})

# Number.of.Engines: drop implausible 0-engine airplanes (likely data errors)
av_data = av_data[(av_data['Number.of.Engines'] != 0) | (av_data['Number.of.Engines'].isna())]

# Purpose.of.flight: merge formatting-only duplicate label
av_data['Purpose.of.flight'] = av_data['Purpose.of.flight'].replace({'Air Race/show': 'Air Race show'})

# --- Broad.phase.of.flight: heavily missing (~85%)
print(av_data.shape)

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
# Inspect
missing_pct = (av_data.isnull().sum() / len(av_data) * 100).sort_values(ascending=False)
print(missing_pct)

7. Dropping columns that are irrelevant to the study or that have large number of missing values. NB kept the Broad.phase.of.flight despite it having 85% missing columns because it is required as one of the metrics for analysis


In [ ]:
# Drop columns that are administrative, not relevant to the safety analysis, 
# or too sparse to support the ~50-example minimum needed for reliable comparisons
cols_to_drop = [
    'Schedule', 'Air.carrier', 'Airport.Code', 'Airport.Name',
    'Report.Status', 'Latitude', 'Longitude', 'FAR.Description',
    'Publication.Date'
]
av_data = av_data.drop(columns=cols_to_drop)

print(av_data.shape)
print(av_data.isnull().sum().sort_values(ascending=False))

In [ ]:
av_data.columns

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [ ]:
# Save cleaned dataset for use in the analysis notebook
av_data.to_csv('data/AviationData_Cleaned.csv', index=False)

print(f"Saved {av_data.shape[0]} rows and {av_data.shape[1]} columns.")